##Run config first

In [0]:
%run "../00_config"

In [0]:
import requests
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import *

##fetch function

In [0]:
def fetch_amazon_products(query: str) -> list:
    url = "https://api.hasdata.com/scrape/amazon/search"
    headers = {
        "x-api-key": HASDATA_AMAZON_SEARCH_KEY,
        "Content-Type": "application/json"
    }
    params = {
        "q":             query,
        "domain":        "www.amazon.sa",
        "language":      "EN"
    }
    try:
        r = requests.get(url, headers=headers, params=params, timeout=15)
        r.raise_for_status()
        products = r.json().get("productResults", [])
        print(f"  '{query}' → {len(products)} products")
        return products
    except Exception as e:
        print(f"  '{query}' → ERROR: {e}")
        return []

## fetch products

In [0]:
all_products = []
 
for category in PRODUCTS_CATEGORIES:
    products = fetch_amazon_products(category)
    for p in products:
        all_products.append({
            "asin":               str(p.get("asin",            "")),
            "product_title":      str(p.get("title",           "")),
            "position":           str(p.get("position",        "")),
            "current_price_sar":  str(p.get("price",           {}).get("currentPrice",  "")),
            "before_price_sar":   str(p.get("price",           {}).get("beforePrice",   "")),
            "currency":           str(p.get("price",           {}).get("symbol",        "SAR")),
            "rating":             str(p.get("reviews",         {}).get("rating",        "")),
            "total_reviews":      str(p.get("reviews",         {}).get("totalReviews",  "")),
            "is_amazon_choice":   str(p.get("badges",          {}).get("amazonChoice",  "")),
            "is_best_seller":     str(p.get("badges",          {}).get("bestSeller",    "")),
            "is_prime":           str(p.get("badges",          {}).get("amazonPrime",   "")),
            "is_sponsored":       str(p.get("isSponsored",     "")),
            "bought_past_month":  str(p.get("boughtInPastMonth", "")),
            "stock_warning":      str(p.get("deliveryInfo",     {}).get("stockWarning","")),
            "product_photo":      str(p.get("image",           "")),
            "product_url":        str(p.get("url",             "")),
            "category":           category,
            "_snapshot_date":     SNAPSHOT_DATE,
            "_ingested_at":       datetime.now(timezone.utc).strftime("%d-%m-%Y %H:%M:%S")
        })
 
print(f"\nTotal products collected: {len(all_products)}")

##Write to Bronze

In [0]:
df_bronze = spark.createDataFrame(all_products)

(df_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE_PRODUCTS)
)

print(f"✅ {df_bronze.count()} products written to {BRONZE_PRODUCTS}")

##Validate

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.bronze.bronze_product;